# Labeling the dataset by a transformer model

Loads the fine-tuned transformer model and labels the **entire** dataset using the
efficient batched method. The model is produced by `transformers_model_finetuner.py`.

## Setup

In [3]:
import pandas as pd
import os

# gfx1150 (AMD Radeon 840M) has no ROCm kernels; present it as gfx1100. Set before torch import; no-op off ROCm.
os.environ.setdefault("HSA_OVERRIDE_GFX_VERSION", "11.0.0")

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm.auto import tqdm

/home/martan/Nienke/Protest_Labelling/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
DATA_DIR = '../data'
MODELS_DIR = '../models'

model_dir = f'{MODELS_DIR}/attempt3-better-keywords-roberta/hf_transformer_model'
TRAIN_CSV = f'{DATA_DIR}/labeled_balanced_20.csv'  # the CSV the model was trained on
INPUT_CSV = f'{DATA_DIR}/labeled.csv'
OUTPUT_CSV = f'{DATA_DIR}/filtered_events_class_with_predicted.csv'

## Loading the model

In [3]:
print(f"\n--- Loading the model from {model_dir} ---")
loaded_tokenizer = AutoTokenizer.from_pretrained(model_dir)
loaded_model = AutoModelForSequenceClassification.from_pretrained(model_dir)

if torch.backends.mps.is_available():
    load_device = torch.device("mps")
elif torch.cuda.is_available():
    load_device = torch.device("cuda")
else:
    load_device = torch.device("cpu")

loaded_model.to(load_device)
loaded_model.eval()

print(f"Using device: {load_device}")
print("Model and tokenizer loaded successfully!")


--- Loading the model from ../models/attempt3-better-keywords-roberta/hf_transformer_model ---


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1373.37it/s]


Using device: cuda
Model and tokenizer loaded successfully!


## Loading the dataset

In [5]:
dataset = pd.read_csv(INPUT_CSV)
len(dataset)

/tmp/ipykernel_623479/4017719280.py:1: DtypeWarning: Columns (0: civilian_targeting) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv(INPUT_CSV)


214232

In [5]:
# Read the class names straight from the model config (set at train time by
# transformers_model_finetuner.py). Fall back to the training CSV only for older
# models saved with generic LABEL_0..N names.
id2label = {int(k): v for k, v in loaded_model.config.id2label.items()}

if all(str(v).startswith("LABEL_") for v in id2label.values()):
    _train_df = pd.read_csv(TRAIN_CSV)
    _train_df = _train_df[~_train_df['class'].isin(['NoN', 'unknown'])]
    id2label = {i: name for i, name in enumerate(sorted(_train_df['class'].unique()))}
    print(f"Model has generic labels; reconstructed {len(id2label)} from {TRAIN_CSV}")
else:
    print(f"Using {len(id2label)} labels from the model config")

assert len(id2label) == loaded_model.config.num_labels
for i in sorted(id2label):
    print(f"  {i}: {id2label[i]}")

Using 20 labels from the model config
  0: animal welfare
  1: blm
  2: climate
  3: culture
  4: discrimination
  5: education
  6: environment
  7: farmers
  8: health care
  9: housing
  10: immigration
  11: labor rights
  12: lgbtq
  13: palestine-israel conflict
  14: pandemic
  15: policies & politics
  16: public services
  17: ukraine-russia war
  18: unjust law enforcement
  19: women rights


## Batched labeling over the whole dataset (fast)

Batched inference with length-sorted dynamic padding, bf16 autocast on CUDA, and
`inference_mode`. Original row order is restored before writing.

In [6]:
MAX_LEN_HF = 128
BATCH_SIZE = 256  # lower this if you hit out-of-memory

texts = [str(t) for t in dataset['clean_notes'].tolist()]
n = len(texts)

order = sorted(range(n), key=lambda i: len(texts[i]))  # group similar lengths for tight padding
predictions = [None] * n

use_bf16 = (load_device.type == 'cuda')

for start in tqdm(range(0, n, BATCH_SIZE), desc="Labeling dataset (batched)"):
    batch_pos = order[start:start + BATCH_SIZE]
    batch_texts = [texts[i] for i in batch_pos]

    inputs = loaded_tokenizer(
        batch_texts,
        return_tensors='pt',
        padding='longest',
        truncation=True,
        max_length=MAX_LEN_HF,
    )
    inputs = {key: val.to(load_device) for key, val in inputs.items()}

    with torch.inference_mode():
        with torch.autocast(device_type=load_device.type, dtype=torch.bfloat16, enabled=use_bf16):
            logits = loaded_model(**inputs).logits
        predicted = torch.argmax(logits, dim=1).cpu().tolist()

    for pos, cls_idx in zip(batch_pos, predicted):
        predictions[pos] = id2label[cls_idx]

dataset['predicted_class'] = predictions
dataset.to_csv(OUTPUT_CSV, index=False)
print(f"Saved predictions to {OUTPUT_CSV}")

Labeling dataset (batched):   0%|          | 0/837 [00:00<?, ?it/s]/home/martan/Nienke/Protest_Labelling/.venv/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:92: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:338.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Labeling dataset (batched): 100%|██████████| 837/837 [23:28<00:00,  1.68s/it]


Saved predictions to ../data/filtered_events_class_with_predicted.csv


## Comparison with the students' model on unseen `unknown` events

The trigger-word-labeled rows all contain the keyword the model trained on, so ~98% there
is mostly leakage. The honest test is the `unknown` events: no trigger word, never trained
on. There's no ground truth, so two samples were hand-labeled (`manual_label`, by Claude)
from the 20-class taxonomy; here we attach **this run's** predictions by `event_id_cnty`:

- **`random_unknown_labeled.csv`** — randomly drawn `unknown` events (unbiased, honest).
- **`manual_labeled_unknown.csv`** — events where the older models *disagreed* (biased; inflates the gap).

Mirrors `research/model_comparison_on_unseen_data/comparison.ipynb`, but scores the model
just labeled above instead of the stored (old-model) predictions.

In [7]:
import plotly.express as px

MANUAL_DIR = f'{DATA_DIR}/manual_labelled_data'

# This run's prediction per event, joined by event id. Use the in-memory dataset
# if the labeling cell ran this session, else read the saved output.
if 'predicted_class' in dataset.columns:
    this_model = dataset[['event_id_cnty', 'predicted_class']]
else:
    this_model = pd.read_csv(OUTPUT_CSV, usecols=['event_id_cnty', 'predicted_class'], low_memory=False)

def load_eval(path):
    d = pd.read_csv(path).drop(columns=['predicted_class_model'])  # stale (old model); use this run instead
    d = d.merge(this_model, on='event_id_cnty', how='left').rename(
        columns={'predicted_class': 'predicted_class_model'})
    d['model_correct'] = d['predicted_class_model'] == d['manual_label']
    d['students_correct'] = d['predicted_class_students'] == d['manual_label']
    d['agree'] = d['predicted_class_model'] == d['predicted_class_students']
    return d

rand = load_eval(f'{MANUAL_DIR}/random_unknown_labeled.csv')      # random unknown events (honest)
disag = load_eval(f'{MANUAL_DIR}/manual_labeled_unknown.csv')     # old-model disagreements (biased)
assert rand['predicted_class_model'].notna().all(), "some events were not found in the labeled dataset"
print(f"random (honest): {len(rand)} | disagreement (biased): {len(disag)}")
print(f"random -> this model {rand['model_correct'].mean():.0%} | students {rand['students_correct'].mean():.0%}")

random (honest): 80 | disagreement (biased): 70
random -> this model 28% | students 50%


In [8]:
n = len(rand)
ma, sa = rand['model_correct'].mean(), rand['students_correct'].mean()
ci = lambda p: 1.96 * (p * (1 - p) / n) ** 0.5  # normal-approx 95% CI (small n!)

head = pd.DataFrame({'source': ['This model', 'Students'],
                     'accuracy': [ma, sa], 'ci': [ci(ma), ci(sa)]})
fig = px.bar(head, x='source', y='accuracy', color='source', text=head['accuracy'].round(3),
             error_y=head['ci'], labels={'source': '', 'accuracy': 'Accuracy on random unknown'})
fig.update_layout(title={'text': f'Honest accuracy on random unknown events (n={n}, 95% CI)', 'x': 0.5},
                  plot_bgcolor='white', width=620, height=430, yaxis={'range': [0, 1]}, showlegend=False)
fig.show()

In [9]:
ag, dis = rand[rand['agree']], rand[~rand['agree']]
print(f"agree on {len(ag)}/{len(rand)} ({len(ag)/len(rand):.0%}); shared label right {ag['model_correct'].mean():.0%}")
print(f"disagreement-only (biased) sample: this model {disag['model_correct'].mean():.0%} vs students {disag['students_correct'].mean():.0%}")

brk = pd.DataFrame({
    'outcome': ['This model right', 'Students right', 'Neither right'],
    'share': [dis['model_correct'].mean(), dis['students_correct'].mean(),
              (~dis['model_correct'] & ~dis['students_correct']).mean()],
})
fig = px.bar(brk, x='outcome', y='share', color='outcome', text=brk['share'].round(3),
             labels={'outcome': '', 'share': 'Share of disagreements'})
fig.update_layout(title={'text': f'When the models disagree on random unknown (n={len(dis)})', 'x': 0.5},
                  plot_bgcolor='white', width=620, height=430, yaxis={'range': [0, 1]}, showlegend=False)
fig.show()

agree on 30/80 (38%); shared label right 57%
disagreement-only (biased) sample: this model 20% vs students 44%


## Diagnostic: is the drop overfitting or collapse?

roberta scores lower on the honest `unknown` set than bert despite the same ~98% on labeled
data. Two checks to tell *why*:

- **(a) Prediction mix on `unknown`** — spread across most of the 20 classes means genuine
  overfitting (poor generalization but still discriminating); piling into 1–2 classes means
  collapse (defaulting to an easy class when no trigger word is present).
- **(b) Three-way honest accuracy** — bert vs roberta vs students on the random sample, to
  size the gap (small n, ±~11pp).

In [10]:
# (a) roberta's prediction mix on the unknown rows: spread across classes => overfitting,
# piled into 1-2 classes => collapse.
if 'predicted_class' in dataset.columns and 'class' in dataset.columns:
    unk = dataset.loc[dataset['class'] == 'unknown', 'predicted_class']
else:
    _d = pd.read_csv(OUTPUT_CSV, usecols=['class', 'predicted_class'], low_memory=False)
    unk = _d.loc[_d['class'] == 'unknown', 'predicted_class']

dist = unk.value_counts(normalize=True).sort_values()
print(f"unknown rows: {len(unk)} | distinct classes predicted: {unk.nunique()}/20 | top class share: {dist.iloc[-1]:.0%}")
fig = px.bar(dist, orientation='h', text=dist.round(3),
             labels={'value': 'share of unknown predictions', 'index': ''})
fig.update_layout(title={'text': f'roberta prediction mix on unknown events (n={len(unk)})', 'x': 0.5},
                  plot_bgcolor='white', width=680, height=520, showlegend=False)
fig.show()

unknown rows: 83921 | distinct classes predicted: 20/20 | top class share: 31%
